## Notebook 1 - Data Validation & Quality Assessment

### Objective
- To validate the structure, coverage, completeness and internal consistency of the aggregated cancer datasets before analytical transformation.

### Input
- Incidence_data_National_Cancer_Group_2026-08-27.csv
- Incidence_data_Region_Cancer_Group_2026-08-27.csv

### Data Source
- NDRS Cancer Incidence and Mortality Dashboard: https://nhsd-ndrs.shinyapps.io/incidence_and_mortality/

1. National dataset: NDRS Cancer Incidence and Mortality Dashboard
    England-level data where each row represents a unique aggregated observation defined by year, gender, age group at diagnosis (Under 1, 1-4, 5-9, 10-14, 15-19, 20-24), and NDRS cancer classification. Coverage: 2013-2022
2. Regional dataset: NDRS Cancer Incidence and Mortality Dashboard
    NHS England region-level data where each row represents a unique aggregated observation defined by year, gender, NHS region, and NDRS cancer classification. Age group aggregated to 0-24. Coverage: 2013-2022

**Flag Definitions from the original dataset:**
- [note1] Age-standardised rates based on numbers lower than 20. The small number of diagnoses may affect the reliability of these rates
- [note2] Age-standardised rates based on numbers lower than 10. Age-specific and non-standardised rates based on numbers lower than 3
- [note3] This note has been removed from this version of the app
- [note4] All Blood cancer and All Head and neck cancer groups by stage at diagnosis have been suppressed due to differing staging systems and a large number of unstageable cancers (All Blood cancers), or due to large differences in types of tumours in the respective detailed cancer groups.
- [note5] For MSOA only. All rates based on numbers between 1 and 7.
- [u] Where the conditions for [note2] and [note4] are met, the rates and confidence interval will display [u]. Where the conditions for [note5] are met, the counts, rates and confidence interval fields will display [u]
For MSOA only. Counts that have not been suppressed have been rounded to the nearest 5.

### Output
- Processed, clean, analysis-ready dataset files in parquet format.

### Import Libraries

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

### Import Dataset

In [2]:
DATA_PATH = Path("../0_data/0_raw")

In [3]:
national_df = pd.read_csv(
    DATA_PATH / "Incidence_data_National_Cancer_Group_2026-08-27.csv"
)

print("File read successfully.")

File read successfully.


In [4]:
regional_df = pd.read_csv(
    DATA_PATH / "Incidence_data_Region_Cancer_Group_2026-08-27.csv"
)

print("File read successfully.")

File read successfully.


## 1. National Incidence Rate Dataset

### 1.1 Initial Inspection

Objective
- Examining the dataset structure

In [5]:
national_df.shape

(15480, 16)

In [6]:
national_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15480 entries, 0 to 15479
Data columns (total 16 columns):
 #   Column                         Non-Null Count  Dtype 
---  ------                         --------------  ----- 
 0   Year                           15480 non-null  int64 
 1   Gender                         15480 non-null  object
 2   Age at diagnosis               15480 non-null  object
 3   Geography type                 15480 non-null  object
 4   Geography code                 15480 non-null  object
 5   Geography name                 15480 non-null  object
 6   Deprivation                    15480 non-null  object
 7   Stage                          15480 non-null  object
 8   NDRS main                      15480 non-null  object
 9   NDRS detailed                  15480 non-null  object
 10  Count                          15480 non-null  int64 
 11  Type of rate                   15480 non-null  object
 12  Rate                           15480 non-null  object
 13  9

In [7]:
national_df.head()

,Year,Gender,Age at diagnosis,Geography type,Geography code,Geography name,Deprivation,Stage,NDRS main,NDRS detailed,Count,Type of rate,Rate,95% lower confidence interval,95% upper confidence interval,Flag
0,2013,Females,0 (Under 1),England,E92000001,England,All quintiles,All stages,Anus,All Anus,0,Age-specific,[u],[u],[u],[note2]
1,2014,Females,0 (Under 1),England,E92000001,England,All quintiles,All stages,Anus,All Anus,0,Age-specific,[u],[u],[u],[note2]
2,2015,Females,0 (Under 1),England,E92000001,England,All quintiles,All stages,Anus,All Anus,0,Age-specific,[u],[u],[u],[note2]
3,2016,Females,0 (Under 1),England,E92000001,England,All quintiles,All stages,Anus,All Anus,0,Age-specific,[u],[u],[u],[note2]
4,2017,Females,0 (Under 1),England,E92000001,England,All quintiles,All stages,Anus,All Anus,0,Age-specific,[u],[u],[u],[note2]


### 1.2 Schema Validation

In [8]:
national_df.columns

Index(['Year', 'Gender', 'Age at diagnosis', 'Geography type',
       'Geography code', 'Geography name ', 'Deprivation', 'Stage',
       'NDRS main', 'NDRS detailed', 'Count', 'Type of rate', 'Rate',
       '95% lower confidence interval', '95% upper confidence interval',
       'Flag'],
      dtype='object')

In [9]:
# Strip schema column name
# Removing the superfluous space at the end in 'Geography name '

national_df.columns = national_df.columns.str.strip()

national_df.columns

Index(['Year', 'Gender', 'Age at diagnosis', 'Geography type',
       'Geography code', 'Geography name', 'Deprivation', 'Stage', 'NDRS main',
       'NDRS detailed', 'Count', 'Type of rate', 'Rate',
       '95% lower confidence interval', '95% upper confidence interval',
       'Flag'],
      dtype='object')

In [10]:
expected_columns = {
    "Year",
    "Gender",
    "Age at diagnosis",
    "Geography type",
    "Geography code",
    "Geography name",
    "Deprivation",
    "Stage",
    "NDRS main",
    "NDRS detailed",
    "Count",
    "Type of rate",
    "Rate",
    "95% lower confidence interval",
    "95% upper confidence interval",
    "Flag"
}

actual_columns = set(national_df.columns)

assert expected_columns == actual_columns

print("Schema consistency confirmed.")

Schema consistency confirmed.


### 1.3 Expected Value Validation

Objective:
- Validating that only the expected values are included in each key dimension

#### 1.3.1 Age Group Validation

In [11]:
# Inspect age group unique values and counts

national_df["Age at diagnosis"].value_counts(dropna=False)

Age at diagnosis
0 (Under 1)    2580
01 to 04       2580
05 to 09       2580
10 to 14       2580
15 to 19       2580
20 to 24       2580
Name: count, dtype: int64

In [12]:
# Check missing values

national_df["Age at diagnosis"].isna().sum()

np.int64(0)

In [13]:
# Validate expected values

expected_age_group = {
    "0 (Under 1)",
    "01 to 04",
    "05 to 09",
    "10 to 14",
    "15 to 19",
    "20 to 24"
}

actual_age_group = set(
    national_df["Age at diagnosis"].dropna().unique()
)

assert actual_age_group == expected_age_group

print("Expected values confirmed.")

Expected values confirmed.


#### 1.3.2 Year Validation

In [14]:
# Inspect year unique values and counts

national_df["Year"].value_counts(dropna=False, sort=False)

Year
2013    1542
2014    1542
2015    1542
2016    1542
2017    1542
2018    1554
2019    1554
2020    1554
2021    1554
2022    1554
Name: count, dtype: int64

In [15]:
# Check missing

national_df["Year"].isna().sum()

np.int64(0)

In [16]:
# Validate expected values

expected_years = set(range(2013, 2023))

actual_years = set(
    national_df["Year"].dropna().unique()
)

assert actual_years == expected_years

print("Expected values confirmed.")

Expected values confirmed.


#### 1.3.3 Gender Validation

In [17]:
# Inspect gender unique values and counts

national_df["Gender"].value_counts(dropna=False)

Gender
Females    7950
Males      7530
Name: count, dtype: int64

In [18]:
# Check missing

national_df["Gender"].isna().sum()

np.int64(0)

In [19]:
# Validate expected values

expected_gender = {"Females", "Males"}

actual_gender = set(
    national_df["Gender"].unique()
)

assert actual_gender == expected_gender

print("Expected values confirmed.")

Expected values confirmed.


#### 1.3.4 Geography Validation

In [20]:
# Inspect geography unique value

national_df["Geography type"].value_counts(dropna=False)

Geography type
England    15480
Name: count, dtype: int64

In [21]:
assert national_df["Geography type"].nunique() == 1
assert national_df["Geography type"].iloc[0] == "England"

print("Expected values confirmed.")

Expected values confirmed.


In [22]:
# Inspect geography unique value

national_df["Geography name"].value_counts(dropna=False)

Geography name
England    15480
Name: count, dtype: int64

In [23]:
assert national_df["Geography name"].nunique() == 1
assert national_df["Geography name"].iloc[0] == "England"

print("Expected values confirmed.")

Expected values confirmed.


#### 1.3.5 Type of Rate Validation

In [24]:
# Inspect type of rate unique value

national_df["Type of rate"].value_counts(dropna=False)

Type of rate
Age-specific    15480
Name: count, dtype: int64

In [25]:
assert national_df["Type of rate"].nunique() == 1
assert national_df["Type of rate"].iloc[0] == "Age-specific"

print("Expected values confirmed.")

Expected values confirmed.


#### 1.3.6 Cancer Group Validation

In [26]:
# Inspect cancer group unique values

national_df["NDRS main"].nunique()

32

In [27]:
# Inspect cancer group unique values

national_df["NDRS detailed"].nunique()

139

In [28]:
national_df["NDRS main"].value_counts()

NDRS main
Blood cancer                                                             2160
Soft tissue sarcoma                                                      2100
Head and neck                                                            1080
Kidney                                                                    840
Bone sarcoma                                                              840
Bladder                                                                   720
Cancer of unknown primary                                                 600
Liver and biliary tract                                                   600
Skin cancer                                                               600
Stomach                                                                   600
Other malignant                                                           600
Brain                                                                     600
Oesophagus                                            

In [29]:
national_df["NDRS detailed"].value_counts()

NDRS detailed
All Anus                                           120
All Bladder                                        120
Bladder - T1 non-muscle-invasive urothelial        120
Bladder - Ta/Tis non-muscle-invasive urothelial    120
Bladder - muscle-invasive urothelial               120
                                                  ... 
All Prostate                                        60
All Testes                                          60
Non-seminoma                                        60
Seminoma                                            60
Testes - other                                      60
Name: count, Length: 139, dtype: int64

### 1.4 Time and Dimension Coverage Validation

Key Finding:
- The number of records per age group increases from 257 to 259 from 2018 onwards.

- Further investigation shows that this is not caused by missing or duplicate records. The change is driven by a classification inconsistency in the 'NDRS detailed' field.

- Before 2018, 'Cardia and oesophagogastric junction' was reported as a single category. From 2018 onwards, it was split into two separate categories: 'Oesophagogastric junction' and 'Cardia'.

- The 'NDRS main' classification remains consistent across the analysis period.

- This classification change should be considered when analysing long-term trends at the detailed cancer type level.

#### 1.4.1 Dimension Coverage Check

In [30]:
# Calculating the counts of each age group by year

year_age_coverage = pd.crosstab(
    national_df["Year"],
    national_df["Age at diagnosis"]
)

year_age_coverage

Age at diagnosis,0 (Under 1),01 to 04,05 to 09,10 to 14,15 to 19,20 to 24
Year,,,,,,
2013,257,257,257,257,257,257
2014,257,257,257,257,257,257
2015,257,257,257,257,257,257
2016,257,257,257,257,257,257
2017,257,257,257,257,257,257
2018,259,259,259,259,259,259
2019,259,259,259,259,259,259
2020,259,259,259,259,259,259
2021,259,259,259,259,259,259


- All year-age combinations were represented in the dataset, indicating complete coverage across the analysis period. However, record counts per age group increased from 257 to 259 from 2018 onwards. This structural change was investigated further to identify whether it resulted from additional cancer categories or changes in another reporting dimension.

#### 1.4.2 Row Combinations Comparison

In [31]:
# Creating 2017 dataframe and 2018 dataframe for comparison

age_group = "01 to 04"

df_2017 = national_df.loc[
    (national_df["Year"] == 2017)
    & (national_df["Age at diagnosis"] == age_group)
].copy()

df_2018 = national_df.loc[
    (national_df["Year"] == 2018)
    & (national_df["Age at diagnosis"] == age_group)
].copy()

In [32]:
# Validation

df_2017.shape

(257, 16)

In [33]:
# Validation

df_2018.shape

(259, 16)

In [34]:
# Creating a list of categorical columns for comparison

categorical_columns = [
    "NDRS main",
    "NDRS detailed"
]

In [35]:
# Calculating the difference

for column in categorical_columns:

    values_2017 = set(df_2017[column].dropna().unique())
    values_2018 = set(df_2018[column].dropna().unique())

    print(f"\n--- {column} ---")

    print(
        "New in 2018:",
        values_2018 - values_2017
    )

    print(
        "Missing in 2018:",
        values_2017 - values_2018
    )


--- NDRS main ---
New in 2018: set()
Missing in 2018: set()

--- NDRS detailed ---
New in 2018: {'Cardia', 'Oesophagogastric junction'}
Missing in 2018: {'Cardia and oesophagogastric junction'}


In [36]:
mask = national_df["NDRS detailed"].isin(
    [
        "Cardia and oesophagogastric junction",
        "Cardia",
        "Oesophagogastric junction"
    ]
)

national_df[mask][
    [
        "Year",
        "Age at diagnosis",
        "NDRS main",
        "NDRS detailed",
        "Count"
    ]
].sort_values(
    ["Year", "Age at diagnosis", "NDRS detailed"]
)

,Year,Age at diagnosis,NDRS main,NDRS detailed,Count
1230,2013,0 (Under 1),Stomach,Cardia and oesophagogastric junction,0
9110,2013,0 (Under 1),Stomach,Cardia and oesophagogastric junction,0
2555,2013,01 to 04,Stomach,Cardia and oesophagogastric junction,0
10365,2013,01 to 04,Stomach,Cardia and oesophagogastric junction,0
3880,2013,05 to 09,Stomach,Cardia and oesophagogastric junction,0
...,...,...,...,...,...
13754,2022,15 to 19,Oesophagus,Oesophagogastric junction,0
7854,2022,20 to 24,Stomach,Cardia,0
15384,2022,20 to 24,Stomach,Cardia,0
7419,2022,20 to 24,Oesophagus,Oesophagogastric junction,1


- The change is driven by a classification inconsistency in the 'NDRS detailed' field. Before 2018, 'Cardia and oesophagogastric junction' was reported as a single category. From 2018 onwards, it was split into two separate categories: 'Oesophagogastric junction' and 'Cardia'.

### 1.5 Expected Grain of Dataset Validation

Key Finding:
- Each row represents a unique aggregated cancer incidence observation defined by the combination of year, gender, age at diagnosis and classification (NDRS main and NDRS detailed).

#### 1.5.1 Checking Uniqueness

In [37]:
# Checking uniqueness in selected dimensions

columns = [
    "Year",
    "Gender",
    "Age at diagnosis",
    "Geography type",
    "Geography code",
    "Geography name",
    "Deprivation",
    "Stage",
    "NDRS main",
    "NDRS detailed"
]

for col in national_df[columns]:
    print(f"\n--- {col} ---")
    print(f"Unique values: {national_df[col].nunique(dropna=False)}")


--- Year ---
Unique values: 10

--- Gender ---
Unique values: 2

--- Age at diagnosis ---
Unique values: 6

--- Geography type ---
Unique values: 1

--- Geography code ---
Unique values: 1

--- Geography name ---
Unique values: 1

--- Deprivation ---
Unique values: 1

--- Stage ---
Unique values: 1

--- NDRS main ---
Unique values: 32

--- NDRS detailed ---
Unique values: 139


In [38]:
# Validating the expected grain of dataset

grain_columns = [
    "Year",
    "Gender",
    "Age at diagnosis",
    "NDRS main",
    "NDRS detailed"
]

national_df.duplicated(
    subset=grain_columns
).sum()

np.int64(0)

### 1.6 Completeness and Missingness

Key Finding:
- All core analytical fields are complete. No standard missing values were identified outside the Flag column. Null values indicate that no warning or suppression condition applies to the corresponding record.

- Diagnosis counts are available for all records, whereas published incidence rates and confidence intervals are unavailable (represented as [u] - indicating the age-specific rates were based on numbers lower than 3 according to the dataset documentation) for 11,869 of 15,480 records.

- All unavailable rate and confidence interval values are systematically associated with [note2], indicating that they are intentionally suppressed under the dataset's published reporting rules rather than representing data quality or accidental missingness.

- As a result, counts can be analysed across the full dataset, while rate-based analysis must be restricted to records with published numerical estimates. Suppressed values will not be interpreted as zero.

#### 1.6.1 Standard missingness

In [39]:
# Calculating the missing values in dataset

national_df.isna().sum()

Year                                0
Gender                              0
Age at diagnosis                    0
Geography type                      0
Geography code                      0
Geography name                      0
Deprivation                         0
Stage                               0
NDRS main                           0
NDRS detailed                       0
Count                               0
Type of rate                        0
Rate                                0
95% lower confidence interval       0
95% upper confidence interval       0
Flag                             3611
dtype: int64

- All core analytical fields are complete. Null values are present only in the Flag column, which indicate that no warning or suppression condition applies according to the original dataset documentation.

#### 1.6.2 Suppressed Values Check

In [40]:
# Counting the number of suppressed values

metric_columns = [
    "Count",
    "Rate",
    "95% lower confidence interval",
    "95% upper confidence interval"
]

for col in metric_columns:
    suppressed = (national_df[col] == "[u]").sum()
    print(f"{col}: {suppressed} suppressed")

Count: 0 suppressed
Rate: 11869 suppressed
95% lower confidence interval: 11869 suppressed
95% upper confidence interval: 11869 suppressed


- Counts are fully available, while rate estimates and their corresponding confidence intervals are unavailable for 11,869 records.

In [41]:
# Distribution of suppressed values among age groups

pd.crosstab(
    national_df["Age at diagnosis"],
    national_df["Rate"] == "[u]"
)

Rate,False,True
Age at diagnosis,,
0 (Under 1),352,2228
01 to 04,423,2157
05 to 09,430,2150
10 to 14,548,2032
15 to 19,782,1798
20 to 24,1076,1504


In [42]:
# Distribution of suppressed values among cancer groups
 
pd.crosstab(
    national_df["NDRS main"],
    national_df["Rate"] == "[u]"
)

Rate,False,True
NDRS main,,
Anus,0,120
Bladder,63,657
Blood cancer,792,1368
Bone sarcoma,291,549
Bowel,115,365
Brain,490,110
Breast,15,105
Cancer of unknown primary,25,575
Cervix,15,45


In [43]:
# Examining Flag x Rate relationship

pd.crosstab(
    national_df["Flag"].fillna("No flag"),
    national_df["Rate"] == "[u]"
)

Rate,False,True
Flag,,
No flag,3611,0
[note2],0,11869


In [44]:
# Validating suppression consistency

rate_unavailable = (
    national_df["Rate"] == "[u]"
)

lower_ci_unavailable = (
    national_df["95% lower confidence interval"] == "[u]"
)

upper_ci_unavailable = (
    national_df["95% upper confidence interval"] == "[u]"
)

assert (rate_unavailable == lower_ci_unavailable).all()
assert (rate_unavailable == upper_ci_unavailable).all()

print("Suppression consistency confirmed.")

Suppression consistency confirmed.


### 1.7 Rate Columns Numeric Conversion

- Converting rates and confidence intervals into numeric values in new columns

In [45]:
# Convert rate and CI columns to numeric, coecring [u] to NaN

rate_columns = [
    "Rate",
    "95% lower confidence interval",
    "95% upper confidence interval"
]

for col in rate_columns:
    national_df[f"{col} (numeric)"] = pd.to_numeric(
        national_df[col], errors="coerce"
    )

In [46]:
# Validation

national_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15480 entries, 0 to 15479
Data columns (total 19 columns):
 #   Column                                   Non-Null Count  Dtype  
---  ------                                   --------------  -----  
 0   Year                                     15480 non-null  int64  
 1   Gender                                   15480 non-null  object 
 2   Age at diagnosis                         15480 non-null  object 
 3   Geography type                           15480 non-null  object 
 4   Geography code                           15480 non-null  object 
 5   Geography name                           15480 non-null  object 
 6   Deprivation                              15480 non-null  object 
 7   Stage                                    15480 non-null  object 
 8   NDRS main                                15480 non-null  object 
 9   NDRS detailed                            15480 non-null  object 
 10  Count                                    15480

In [47]:
# Confirm NaN in numeric columns corresponds exactly to [u] in original
for col in rate_columns:
    original_suppressed = (national_df[col] == "[u]")
    numeric_nan = national_df[f"{col} (numeric)"].isna()
    assert (original_suppressed == numeric_nan).all(), \
        f"Mismatch between suppressed values and NaN in {col} (numeric)"

print("Numeric conversion validation confirmed.")

Numeric conversion validation confirmed.


- Rate and confidence interval columns have been converted to numeric equivalents with [u] values coerced to NaN. The conversion has been validated — all NaN values in numeric columns correspond exactly to suppressed [u] values in the original columns. Original string columns are retained as an audit trail.

## 2. Regional Incidence Rate Dataset

### 2.1 Initial Inspection

- Examine the dataset structure

In [48]:
regional_df.shape

(18090, 16)

In [49]:
regional_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 18090 entries, 0 to 18089
Data columns (total 16 columns):
 #   Column                         Non-Null Count  Dtype 
---  ------                         --------------  ----- 
 0   Year                           18090 non-null  int64 
 1   Gender                         18090 non-null  object
 2   Age at diagnosis               18090 non-null  object
 3   Geography type                 18090 non-null  object
 4   Geography code                 18090 non-null  object
 5   Geography name                 18090 non-null  object
 6   Deprivation                    18090 non-null  object
 7   Stage                          18090 non-null  object
 8   NDRS main                      18090 non-null  object
 9   NDRS detailed                  18090 non-null  object
 10  Count                          18090 non-null  int64 
 11  Type of rate                   18090 non-null  object
 12  Rate                           18090 non-null  object
 13  9

In [50]:
regional_df.head()

,Year,Gender,Age at diagnosis,Geography type,Geography code,Geography name,Deprivation,Stage,NDRS main,NDRS detailed,Count,Type of rate,Rate,95% lower confidence interval,95% upper confidence interval,Flag
0,2013,Females,00 to 24,Government Office Region,E12000004,East Midlands,All quintiles,All stages,Anus,All Anus,0,Age-specific,[u],[u],[u],[note2]
1,2014,Females,00 to 24,Government Office Region,E12000004,East Midlands,All quintiles,All stages,Anus,All Anus,0,Age-specific,[u],[u],[u],[note2]
2,2015,Females,00 to 24,Government Office Region,E12000004,East Midlands,All quintiles,All stages,Anus,All Anus,0,Age-specific,[u],[u],[u],[note2]
3,2016,Females,00 to 24,Government Office Region,E12000004,East Midlands,All quintiles,All stages,Anus,All Anus,0,Age-specific,[u],[u],[u],[note2]
4,2017,Females,00 to 24,Government Office Region,E12000004,East Midlands,All quintiles,All stages,Anus,All Anus,0,Age-specific,[u],[u],[u],[note2]


### 2.2 Schema Validation

In [51]:
regional_df.columns

Index(['Year', 'Gender', 'Age at diagnosis', 'Geography type',
       'Geography code', 'Geography name ', 'Deprivation', 'Stage',
       'NDRS main', 'NDRS detailed', 'Count', 'Type of rate', 'Rate',
       '95% lower confidence interval', '95% upper confidence interval',
       'Flag'],
      dtype='object')

In [52]:
# Strip schema column name
# Removing the superfluous space at the end in 'Geography name '

regional_df.columns = regional_df.columns.str.strip()

regional_df.columns

Index(['Year', 'Gender', 'Age at diagnosis', 'Geography type',
       'Geography code', 'Geography name', 'Deprivation', 'Stage', 'NDRS main',
       'NDRS detailed', 'Count', 'Type of rate', 'Rate',
       '95% lower confidence interval', '95% upper confidence interval',
       'Flag'],
      dtype='object')

In [53]:
expected_columns_reg = {
    "Year",
    "Gender",
    "Age at diagnosis",
    "Geography type",
    "Geography code",
    "Geography name",
    "Deprivation",
    "Stage",
    "NDRS main",
    "NDRS detailed",
    "Count",
    "Type of rate",
    "Rate",
    "95% lower confidence interval",
    "95% upper confidence interval",
    "Flag"
}

actual_columns_reg = set(regional_df.columns)

assert actual_columns_reg == expected_columns_reg

print("Schema consistency confirmed.")

Schema consistency confirmed.


### 2.3 Expected Value Validation

- Validating only the expected values are included in the key dimension columns

#### 2.3.1 Age Group Validation

In [54]:
# Inspect age group unique value and counts

regional_df["Age at diagnosis"].value_counts(dropna=False)

Age at diagnosis
00 to 24    18090
Name: count, dtype: int64

In [55]:
# Check missing

regional_df["Age at diagnosis"].isna().sum()

np.int64(0)

In [56]:
# Validation

assert regional_df["Age at diagnosis"].dropna().unique() == "00 to 24"

print("Expected values confirmed.")

Expected values confirmed.


#### 2.3.2 Year Validation

In [57]:
# Inspect year unique value and counts

regional_df["Year"].value_counts(dropna=False, sort=False)

Year
2013    1800
2014    1800
2015    1800
2016    1800
2017    1800
2018    1818
2019    1818
2020    1818
2021    1818
2022    1818
Name: count, dtype: int64

In [58]:
# Check missing

regional_df["Year"].isna().sum()

np.int64(0)

In [59]:
# Validation

expected_years_reg = set(range(2013,2023))

actual_years_reg = set(regional_df["Year"].dropna().unique())

assert actual_years_reg == expected_years_reg

print("Expected values confirmed.")

Expected values confirmed.


#### 2.3.3 Gender Validation

In [60]:
# Inspect gender unique value and counts

regional_df["Gender"].value_counts(dropna=False)

Gender
Females    9315
Males      8775
Name: count, dtype: int64

In [61]:
# Check missing

regional_df["Gender"].isna().sum()

np.int64(0)

In [62]:
# Validation

expected_gender_reg = {
    "Females",
    "Males"
}

actual_gender_reg = set(regional_df["Gender"].dropna().unique())

assert actual_gender_reg == expected_gender_reg

print("Expected values confirmed.")

Expected values confirmed.


#### 2.3.4 Geography Validation

Key finding:

- The regional dataset uses 9 NHS England legacy regions reflecting the pre-2019 administrative structure, instead of the reorganised 7 NHS England regions in 2019. This structure is retained across the full 2013–2022 analysis period for consistency. Direct comparison with publications using the current seven-region structure should be made with caution.

In [63]:
# Inspect geogrpahy unique value and counts

regional_df["Geography type"].value_counts(dropna=False)

Geography type
Government Office Region    18090
Name: count, dtype: int64

In [64]:
regional_df["Geography code"].value_counts(dropna=False)

Geography code
E12000004    2010
E12000006    2010
E12000007    2010
E12000001    2010
E12000002    2010
E12000008    2010
E12000009    2010
E12000005    2010
E12000003    2010
Name: count, dtype: int64

In [65]:
regional_df["Geography name"].value_counts(dropna=False)

Geography name
East Midlands               2010
East of England             2010
London                      2010
North East                  2010
North West                  2010
South East                  2010
South West                  2010
West Midlands               2010
Yorkshire and The Humber    2010
Name: count, dtype: int64

- The regional dataset uses 9 NHS England legacy regions reflecting the pre-2019 administrative structure, instead of the reorganised 7 NHS England regions in 2019. This structure is retained across the full 2013–2022 analysis period for consistency. Direct comparison with publications using the current seven-region structure should be made with caution.

In [66]:
# Check missing

regional_df["Geography type"].isna().sum()

np.int64(0)

In [67]:
regional_df["Geography code"].isna().sum()

np.int64(0)

In [68]:
regional_df["Geography name"].isna().sum()

np.int64(0)

In [69]:
# Validation

expected_regions = {
    "East Midlands",
    "East of England",
    "London",
    "North East",
    "North West",
    "South East",
    "South West",
    "West Midlands",
    "Yorkshire and The Humber"
}

actual_regions = set(regional_df["Geography name"].dropna().unique())

assert actual_regions == expected_regions

print("Expected values confirmed.")

Expected values confirmed.


#### 2.3.5 Regional Coverage Over Years

- The same NDRS detailed classification change identified in the national dataset is present here: From 2018 onwards, the category 'Cardia and oesophagogastric junction' was split into two separate categories: 'Oesophagogastric junction' and 'Cardia', while the 'NDRS main' classification remains consistent. 

- This should be considered when analysing detailed cancer type trends across the full 2013-2022 period.

In [70]:
# Validate consistency in count per region over analysis years

pd.crosstab(
    regional_df["Year"],
    regional_df["Geography name"]
)

Geography name,East Midlands,East of England,London,North East,North West,South East,South West,West Midlands,Yorkshire and The Humber
Year,,,,,,,,,
2013,200,200,200,200,200,200,200,200,200
2014,200,200,200,200,200,200,200,200,200
2015,200,200,200,200,200,200,200,200,200
2016,200,200,200,200,200,200,200,200,200
2017,200,200,200,200,200,200,200,200,200
2018,202,202,202,202,202,202,202,202,202
2019,202,202,202,202,202,202,202,202,202
2020,202,202,202,202,202,202,202,202,202
2021,202,202,202,202,202,202,202,202,202


- The record count increases from 200 to 202 from 2018 onwards. This might due to the inconsistency in NDRS detailed cancer classification from 2018 onwards. Further investigation is needed to verify whether this hypothesis is true.

In [71]:
# Creating 2017 dataframe and 2018 dataframe for comparison

region = "London"

reg_df_2017 = regional_df.loc[
    (regional_df["Year"] == 2017)
    & (regional_df["Geography name"] == region)
].copy()

reg_df_2018 = regional_df.loc[
    (regional_df["Year"] == 2018)
    & (regional_df["Geography name"] == region)
].copy()

In [72]:
# Validation

reg_df_2017.shape

(200, 16)

In [73]:
reg_df_2018.shape

(202, 16)

In [74]:
# Creating a list of categorical columns for comparison

categorical_columns = [
    "NDRS main",
    "NDRS detailed"
]

# Calculating the difference

for column in categorical_columns:

    values_2017 = set(reg_df_2017[column].dropna().unique())
    values_2018 = set(reg_df_2018[column].dropna().unique())

    print(f"\n--- {column} ---")

    print(
        "New in 2018:",
        values_2018 - values_2017
    )

    print(
        "Missing in 2018:",
        values_2017 - values_2018
    )


--- NDRS main ---
New in 2018: set()
Missing in 2018: set()

--- NDRS detailed ---
New in 2018: {'Cardia', 'Oesophagogastric junction'}
Missing in 2018: {'Cardia and oesophagogastric junction'}


- The same NDRS detailed classification change identified in the national dataset is present here: From 2018 onwards, the category 'Cardia and oesophagogastric junction' was split into two separate categories: 'Oesophagogastric junction' and 'Cardia', while the 'NDRS main' classification remains consistent. This should be considered when analysing detailed cancer type trends across the full 2013-2022 period.

#### 2.3.6 Type of Rate Validation

In [75]:
# Inspect unique value and count

regional_df["Type of rate"].value_counts(dropna=False)

Type of rate
Age-specific    18090
Name: count, dtype: int64

In [76]:
# Check missing

regional_df["Type of rate"].isna().sum()

np.int64(0)

In [77]:
# Validation

assert regional_df["Type of rate"].dropna().unique() == "Age-specific"

print("Expected values confirmed.")

Expected values confirmed.


#### 2.3.7 Cancer Group Validation

Key Finding:
- The regional dataset contains 31 NDRS main and 110 NDRS detailed categories, compared to 32 and 133 respectively in the national dataset. The reduced coverage in the regional dataset suggest the  reflection of NDRS data governance policy: cancer types with consistently small case counts at regional level are excluded from publication to prevent unreliable estimates and protect patient confientiality.

- The national dataset uses stage-specific subcategories for bladder cancers (T1, Ta/Tis, muscle-invasive, etc.), while the regional dataset uses different groupings (malignant or in situ / uncertain or unknown) that aggregate across stages. Both approaches describe the same underlying cancer group. The renal pelvis and ureter group follows the same pattern.

- As the two datasets are analysed separately throughout this project, the inconsistency in cancer group coverage does not affecte the analysis. Where cancer types are absent from the regional dataset, they are not assumed to have zero incidence - they are simply not reportable at this geographic granularity.

In [78]:
# Inspect unique value

regional_df["NDRS main"].nunique()

31

In [79]:
regional_df["NDRS detailed"].nunique()

110

In [80]:
regional_main = set(regional_df["NDRS main"].dropna().unique())
national_main = set(national_df["NDRS main"].dropna().unique())

print(
    "Missing in national:",
    regional_main - national_main
    )

print(
    "Missing in regional:",
    national_main - regional_main
)

Missing in national: {'Heart, mediastinum, pleura, other and ill-defined'}
Missing in regional: {'Heart, mediastinum, pleura, other and ill-defined respiratory disease', 'Other malignant'}


In [81]:
regional_detailed = set(regional_df["NDRS detailed"].dropna().unique())
national_detailed = set(national_df["NDRS detailed"].dropna().unique())

print(
    "Missing in national:",
    regional_detailed - national_detailed
    )

print(
    "Missing in regional:",
    national_detailed - regional_detailed
)

Missing in national: {'Bladder - uncertain or unknown', 'Bladder - malignant or in situ', 'Renal pelvis and ureter - uncertain or unknown', 'All Heart, mediastinum, pleura, other and ill-defined', 'Renal pelvis and ureter - malignant or in situ'}
Missing in regional: {'Bladder - other morphology or uncertain/unknown behaviour', 'Myofibrosarcomas and other fibroblastic sarcomas', 'Bladder - T1 non-muscle-invasive urothelial', 'Synovial', 'All Other malignant', 'Undifferentiated Sarcoma', 'Bone tumours of intermediate behaviour', 'Malignant peripheral nerve sheath tumour (MPNST)', 'Soft tissue tumours of intermediate behaviour', 'Liposarcoma', 'Phyllodes', 'Vascular Tumours', 'All Heart, mediastinum, pleura, other and ill-defined respiratory disease', 'Bladder - Ta/Tis non-muscle-invasive urothelial', 'Other and ill-defined sites', 'Bladder - muscle-invasive urothelial', 'Chondrosarcoma', 'Gastrointestinal stromal sarcoma (GIST)', 'Other malignant bone tumours', 'Placenta', 'Ewing sarcom

- The regional dataset contains 31 NDRS main and 110 NDRS detailed categories, compared to 32 and 133 respectively in the national dataset. The reduced coverage in the regional dataset suggest the  reflection of NDRS data governance policy: cancer types with consistently small case counts at regional level are excluded from publication to prevent unreliable estimates and protect patient confientiality.

- One label difference was identified: 'Heart, mediastinum, pleura, other and ill-defined respiratory disease' in the national dataset corresponds to 'Heart, mediastinum, pleura, other and ill-defined' in the regional dataset. These are interpreted as equivalent categories referring to the same cancer group.

- As the two datasets are analysed separately throughout this project, the inconsistency in cancer group coverage does not affecte the analysis. Where cancer types are absent from the regional dataset, they are not assumed to have zero incidence - they are simply not reportable at this geographic granularity.

In [82]:
# Check regional categories not in national
# (excluding known label variant)

unexpected = regional_detailed - national_detailed - {
    "All Heart, mediastinum, pleura, other and ill-defined"
}

print("Regional categories not in national (unexpected): ", unexpected)

Regional categories not in national (unexpected):  {'Bladder - malignant or in situ', 'Renal pelvis and ureter - malignant or in situ', 'Renal pelvis and ureter - uncertain or unknown', 'Bladder - uncertain or unknown'}


In [83]:
regional_df[regional_df["NDRS detailed"].isin([
    "Renal pelvis and ureter - malignant or in situ",
    "Bladder - malignant or in situ",
    "Bladder - uncertain or unknown",
    "Renal pelvis and ureter - uncertain or unknown"
])]["NDRS main"].unique()

array(['Bladder', 'Renal pelvis and ureter'], dtype=object)

In [84]:
national_df[national_df["NDRS main"] == "Bladder"]["NDRS detailed"].unique()

array(['All Bladder', 'Bladder - T1 non-muscle-invasive urothelial',
       'Bladder - Ta/Tis non-muscle-invasive urothelial',
       'Bladder - muscle-invasive urothelial',
       'Bladder - other morphology or uncertain/unknown behaviour',
       'Bladder - unknown stage urothelial'], dtype=object)

In [85]:
national_df[national_df["NDRS main"] == "Renal pelvis and ureter"]["NDRS detailed"].unique()

array(['All Renal pelvis and ureter'], dtype=object)

In [86]:
regional_df[regional_df["NDRS detailed"].isin([
    "Renal pelvis and ureter - malignant or in situ",
    "Bladder - malignant or in situ",
    "Bladder - uncertain or unknown",
    "Renal pelvis and ureter - uncertain or unknown"
])]["Count"].describe()

count    720.000000
mean       0.363889
std        0.857480
min        0.000000
25%        0.000000
50%        0.000000
75%        0.000000
max        7.000000
Name: Count, dtype: float64

- The four categories present in the regional dataset but absent from the national dataset: 'Bladder - malignant or in situ', 'Bladder - uncertain or unknown', 'Renal pelvis and ureter - malignant or in situ', and 'Renal pelvis and ureter - uncertain or unknown' suggest a structural classification difference between the two dataset exports rather than genuinely different cancer populations.

- The national dataset uses stage-specific subcategories for bladder cancers (T1, Ta/Tis, muscle-invasive, etc.), while the regional dataset uses different groupings (malignant or in situ / uncertain or unknown) that aggregate across stages. Both approaches describe the same underlying cancer group. The renal pelvis and ureter group follows the same pattern.

- Case counts for these categories are negligible in the 0–24 age group (mean: 0.36 per record, max: 7, median: 0), consistent with the known rarity of bladder and renal pelvic cancers in children and young people. As both datasets are analysed separately and these categories represent a very small fraction of CYP cancer burden, this classification difference does not materially affect the analysis. These categories are retained in the regional dataset without modification.

### 2.4 Expected Grain of Dataset Validation

Key Finding:
- Each row represents a unique aggregated cancer incidence observation defined by the combination of year, gender, NHS region, and cancer classification (NDRS main and NDRS detailed). Age group aggregated to 0-24. No duplicate rows were identified on this grain.

In [87]:
# Checking uniqueness in selected dimensions

columns = [
    "Year",
    "Gender",
    "Age at diagnosis",
    "Geography type",
    "Geography code",
    "Geography name",
    "Deprivation",
    "Stage",
    "NDRS main",
    "NDRS detailed"
]

for col in regional_df[columns]:
    print(f"\n--- {col} ---")
    print(f"Unique values: {regional_df[col].nunique(dropna=False)}")


--- Year ---
Unique values: 10

--- Gender ---
Unique values: 2

--- Age at diagnosis ---
Unique values: 1

--- Geography type ---
Unique values: 1

--- Geography code ---
Unique values: 9

--- Geography name ---
Unique values: 9

--- Deprivation ---
Unique values: 1

--- Stage ---
Unique values: 1

--- NDRS main ---
Unique values: 31

--- NDRS detailed ---
Unique values: 110


In [88]:
# Define expected grain for regional dataset

grain_columns_reg = [
    "Year",
    "Gender",
    "Age at diagnosis",
    "Geography type",
    "Geography code",
    "Geography name",
    "NDRS main",
    "NDRS detailed"
]

regional_df.duplicated(
    subset=grain_columns_reg
).sum()

np.int64(0)

- Each row represents a unique aggregated cancer incidence observation defined by the combination of year, gender, NHS region, and cancer classification (NDRS main and NDRS detailed). Age group aggregated to 0-24. No duplicate rows were identified on this grain.

### 2.5 Completeness and Missingness

#### 2.5.1 Standard Missingness

In [89]:
regional_df.isna().sum()

Year                                0
Gender                              0
Age at diagnosis                    0
Geography type                      0
Geography code                      0
Geography name                      0
Deprivation                         0
Stage                               0
NDRS main                           0
NDRS detailed                       0
Count                               0
Type of rate                        0
Rate                                0
95% lower confidence interval       0
95% upper confidence interval       0
Flag                             4798
dtype: int64

#### 2.5.2 Suppressed Values

In [90]:
metric_columns = [
    "Count",
    "Rate",
    "95% lower confidence interval",
    "95% upper confidence interval"
]

for col in metric_columns:
    suppressed = (regional_df[col] == "[u]").sum()
    print(f"{col}: {suppressed} suppressed")

Count: 0 suppressed
Rate: 13292 suppressed
95% lower confidence interval: 13292 suppressed
95% upper confidence interval: 13292 suppressed


- Counts are fully available, while rate estimates and their corresponding confidence intervals are unavailable for 13,292 records.

#### 2.5.3 Suppression Consistency Validation

In [91]:
rate_suppressed = (regional_df["Rate"] == "[u]")
lower_suppressed = (regional_df["95% lower confidence interval"] == "[u]")
upper_suppressed = (regional_df["95% upper confidence interval"] == "[u]")

assert (rate_suppressed == lower_suppressed).all()
assert (rate_suppressed == upper_suppressed).all()
print("Suppression consistency confirmed.")

Suppression consistency confirmed.


#### 2.5.4 Distribution of Suppression Across Regions and Cancer Groups

In [92]:
pd.crosstab(
    regional_df["Geography name"],
    regional_df["Rate"] == "[u]"
)

Rate,False,True
Geography name,,
East Midlands,466,1544
East of England,539,1471
London,663,1347
North East,355,1655
North West,589,1421
South East,637,1373
South West,500,1510
West Midlands,525,1485
Yorkshire and The Humber,524,1486


In [93]:
pd.crosstab(
    regional_df["NDRS main"],
    regional_df["Rate"] == "[u]"
)

Rate,False,True
NDRS main,,
Anus,0,180
Bladder,55,485
Blood cancer,1148,2092
Bone sarcoma,174,6
Bowel,181,539
Brain,684,216
Breast,54,126
Cancer of unknown primary,11,889
Cervix,57,33


Key Finding:
- All core analytical fields are complete. Null values are present only in the Flag column, consistent with the national dataset.

- 13,292 records have suppressed rates and confidence intervals, represented as [u], where the underlying count falls below the NHS Digital disclosure threshold. 

- Counts are available for all records. The distribution of suppression across regions and cancer types is shown above. As with the national dataset, suppressed rates are not interpreted as zero and rate-based analysis is restricted to records with published numerical estimates.

### 2.6 Rate Columns Numeric Conversion

- Converting rates and confidence intervals into numeric values in new columns

In [94]:
# Convert rate and CI columns to numeric, coercing [u] to NaN

rate_columns = [
    "Rate",
    "95% lower confidence interval",
    "95% upper confidence interval"
]

for col in rate_columns:
    regional_df[f"{col} (numeric)"] = pd.to_numeric(
        regional_df[col], errors="coerce"
    )

In [95]:
# Validation

regional_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 18090 entries, 0 to 18089
Data columns (total 19 columns):
 #   Column                                   Non-Null Count  Dtype  
---  ------                                   --------------  -----  
 0   Year                                     18090 non-null  int64  
 1   Gender                                   18090 non-null  object 
 2   Age at diagnosis                         18090 non-null  object 
 3   Geography type                           18090 non-null  object 
 4   Geography code                           18090 non-null  object 
 5   Geography name                           18090 non-null  object 
 6   Deprivation                              18090 non-null  object 
 7   Stage                                    18090 non-null  object 
 8   NDRS main                                18090 non-null  object 
 9   NDRS detailed                            18090 non-null  object 
 10  Count                                    18090

In [96]:
# Confirm NaN in numeric columns corresponds exactly to [u] in original
for col in rate_columns:
    original_suppressed = (regional_df[col] == "[u]")
    numeric_nan = regional_df[f"{col} (numeric)"].isna()
    assert (original_suppressed == numeric_nan).all(), \
        f"Mismatch between suppressed values and NaN in {col} (numeric)"

print("Numeric conversion validation confirmed.")

Numeric conversion validation confirmed.


- Rate and confidence interval columns have been converted to numeric equivalents with [u] values coerced to NaN. The conversion has been validated — all NaN values in numeric columns correspond exactly to suppressed [u] values in the original columns. Original string columns are retained as an audit trail.

## 3. Parquet Output

In [97]:
# Define output path

OUTPUT_PATH = Path("../0_data/1_validated")


# National dataset

national_df.to_parquet(
    OUTPUT_PATH / "national_validated.parquet", index=False
)


# Regional dataset

regional_df.to_parquet(
    OUTPUT_PATH / "regional_validated.parquet", index=False
)

In [98]:
# Validation

test_national = pd.read_parquet(
    OUTPUT_PATH / "national_validated.parquet"
)

assert test_national.shape == national_df.shape
assert list(test_national.columns) == list(national_df.columns)

print("National dataset output validated.")

National dataset output validated.


In [99]:
test_regional = pd.read_parquet(
    OUTPUT_PATH / "regional_validated.parquet"
)

assert test_regional.shape == regional_df.shape
assert list(test_regional.columns) == list(regional_df.columns)

print("Regional dataset output validated.")

Regional dataset output validated.
